In [ ]:
import pandas as pd
import numpy as np
import timeit

def load_and_clean_power_data():
    # Читаємо файл. Роздільник тут крапка з комою (;), а пропуски позначені знаком питання (?)
    df_power = pd.read_csv('household_power_consumption.txt', sep=';', 
                           low_memory=False, na_values=['?'])
    
    # Видаляємо всі рядки, де є порожні клітинки
    df_power = df_power.dropna()
    
    # Об'єднуємо дату і час в один правильний стовпчик Datetime
    df_power['Datetime'] = pd.to_datetime(df_power['Date'] + ' ' + df_power['Time'], dayfirst=True)
    
    # Перетворюємо всі числові стовпці з тексту на числа
    numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    for col in numeric_cols:
        df_power[col] = pd.to_numeric(df_power[col])
        
    return df_power

# Запускаємо функцію і зберігаємо результат у змінну df_p
df_p = load_and_clean_power_data()
print("Датасет успішно завантажено та очищено!")
df_p.head()

In [ ]:
def query_1(df):
    return df[df['Global_active_power'] > 5.0]

# Заміряємо час: запускаємо функцію 5 разів і беремо середнє значення
execution_time_1 = timeit.timeit(lambda: query_1(df_p), number=5) / 5
print(f"Час виконання Завдання 1: {execution_time_1:.5f} секунд")
print(f"Знайдено рядків: {len(query_1(df_p))}")
display(query_1(df_p).head(3))

In [ ]:
def query_2(df):
    return df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20) & 
              (df['Sub_metering_2'] > df['Sub_metering_3'])]

execution_time_2 = timeit.timeit(lambda: query_2(df_p), number=5) / 5
print(f"Час виконання Завдання 2: {execution_time_2:.5f} секунд")
print(f"Знайдено рядків: {len(query_2(df_p))}")
display(query_2(df_p).head(3))

In [ ]:
def query_4(df):
    filtered = df[(df['Datetime'].dt.hour >= 18) & (df['Global_active_power'] > 6.0)]
    filtered = filtered[(filtered['Sub_metering_2'] > filtered['Sub_metering_1']) & 
                        (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])]
    
    half = len(filtered) // 2
    part1 = filtered.iloc[:half:3]
    part2 = filtered.iloc[half::4]
    return pd.concat([part1, part2])

execution_time_4 = timeit.timeit(lambda: query_4(df_p), number=5) / 5
print(f"Час виконання Завдання 4: {execution_time_4:.5f} секунд")
display(query_4(df_p).head(5))

In [ ]:
df_stats = df_p.copy()

# Нормалізація (Min-Max)
min_val = df_stats['Global_active_power'].min()
max_val = df_stats['Global_active_power'].max()
df_stats['GAP_Normalized'] = (df_stats['Global_active_power'] - min_val) / (max_val - min_val)

# Стандартизація (Z-score)
mean_val = df_stats['Voltage'].mean()
std_val = df_stats['Voltage'].std()
df_stats['Voltage_Standardized'] = (df_stats['Voltage'] - mean_val) / std_val

print("Результат математичної нормалізації:")
display(df_stats[['Global_active_power', 'GAP_Normalized', 'Voltage', 'Voltage_Standardized']].head())

# Коефіцієнти кореляції
pearson = df_stats['Global_active_power'].corr(df_stats['Global_intensity'], method='pearson')
spearman = df_stats['Global_active_power'].corr(df_stats['Global_intensity'], method='spearman')
print(f"\nКоефіцієнт Пірсона: {pearson:.5f}")
print(f"Коефіцієнт Спірмена: {spearman:.5f}")

In [1]:
!pip install scipy
